# 面试问题：FP8 格式与 Delayed Scaling 应该怎样实现和验证？

可直接复述的回答：FP8 不是一个格式，E4M3 精度更高但范围较小，E5M2 范围更大但尾数更少。训练通常保留高精度主权重，只把算子输入输出量化。Scale 把真实张量范围映射到 FP8 可表示范围，选择过小会饱和，过大会损失小值精度。Delayed Scaling 使用历史 amax，减少每步同步和抖动，但遇到突发峰值会滞后。系统必须记录饱和率、量化误差和 amax history，并为异常峰值提供当前批次 fallback。格式、scale recipe 与真实 Tensor Core kernel 是三件不同的事。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：LLM MLP 激活批次与输入预览

六个批次模拟客服摘要模型 `mlp_up` 激活，保留 step、层名和可读统计。第四步包含长文档导致的异常峰值；这是离线张量 fixture，不代表真实模型分布。


In [1]:
import numpy as np  # 使用 NumPy 基础运算模拟 FP8 数值路径。
batches01 = [  # 构造连续训练步骤的激活样本。
    {"step": 1, "layer": "mlp_up", "values": [-2.4, -0.8, -0.1, 0.2, 1.1, 2.0]},  # 正常激活范围。
    {"step": 2, "layer": "mlp_up", "values": [-2.8, -1.0, -0.2, 0.4, 1.4, 2.7]},  # amax 小幅增加。
    {"step": 3, "layer": "mlp_up", "values": [-3.3, -1.2, -0.3, 0.6, 1.8, 3.1]},  # 正常训练波动。
    {"step": 4, "layer": "mlp_up", "values": [-26.0, -2.0, -0.4, 0.8, 2.2, 28.0]},  # 长文档触发异常峰值。
    {"step": 5, "layer": "mlp_up", "values": [-3.0, -1.1, -0.2, 0.5, 1.5, 2.9]},  # 峰值后恢复正常。
    {"step": 6, "layer": "mlp_up", "values": [-3.5, -1.4, -0.4, 0.7, 1.9, 3.4]},  # 新的正常范围。
]  # 完成六步可读激活记录。
print("教学实验输入：step | layer | amax | values")  # 输出输入预览表头。
for batch01 in batches01:  # 逐批展示激活范围。
    values01 = np.array(batch01["values"], dtype=np.float32)  # 转换为可计算张量。
    print(batch01["step"], batch01["layer"], round(float(np.abs(values01).max()), 2), values01.tolist())  # 输出批次 amax 和原始值。


教学实验输入：step | layer | amax | values
1 mlp_up 2.4 [-2.4000000953674316, -0.800000011920929, -0.10000000149011612, 0.20000000298023224, 1.100000023841858, 2.0]
2 mlp_up 2.8 [-2.799999952316284, -1.0, -0.20000000298023224, 0.4000000059604645, 1.399999976158142, 2.700000047683716]
3 mlp_up 3.3 [-3.299999952316284, -1.2000000476837158, -0.30000001192092896, 0.6000000238418579, 1.7999999523162842, 3.0999999046325684]
4 mlp_up 28.0 [-26.0, -2.0, -0.4000000059604645, 0.800000011920929, 2.200000047683716, 28.0]
5 mlp_up 3.0 [-3.0, -1.100000023841858, -0.20000000298023224, 0.5, 1.5, 2.9000000953674316]
6 mlp_up 3.5 [-3.5, -1.399999976158142, -0.4000000059604645, 0.699999988079071, 1.899999976158142, 3.4000000953674316]


## 2. Baseline（基线）：第一批固定 Scale

基线只用第一批 amax 设置 E4M3 scale，后续不更新。教学量化器保留 3 位尾数并裁剪到 E4M3 最大有限值 448，用于观察误差和饱和，不冒充真实位级编码。


In [2]:
def fake_fp8_01(values01, scale01, max_code01=448.0, mantissa_bits01=3):  # 实现可解释的 FP8 舍入与裁剪。
    scaled01 = values01 / scale01  # 把真实值映射到 FP8 数值域。
    saturation01 = np.abs(scaled01) > max_code01  # 标记超出有限范围的元素。
    clipped01 = np.clip(scaled01, -max_code01, max_code01)  # 对超范围值执行饱和裁剪。
    magnitude01 = np.abs(clipped01)  # 提取绝对值计算局部量化步长。
    exponent01 = np.floor(np.log2(np.maximum(magnitude01, 2.0 ** -10)))  # 估计每个值的二进制指数。
    step01 = np.power(2.0, exponent01 - mantissa_bits01)  # 根据尾数位计算舍入步长。
    rounded01 = np.sign(clipped01) * np.round(magnitude01 / step01) * step01  # 模拟浮点尾数舍入。
    restored01 = rounded01 * scale01  # 映射回真实激活尺度。
    return restored01.astype(np.float32), float(saturation01.mean())  # 返回反量化值和饱和率。
first_amax01 = max(abs(value01) for value01 in batches01[0]["values"])  # 读取第一批绝对最大值。
static_scale01 = first_amax01 / 448.0  # 使用首批 amax 设置固定 E4M3 scale。
baseline_rows01 = []  # 收集固定 scale 的逐步误差。
for batch01 in batches01:  # 对全部训练步骤复用固定 scale。
    values01 = np.array(batch01["values"], dtype=np.float32)  # 读取当前激活。
    restored01, saturation01 = fake_fp8_01(values01, static_scale01)  # 执行固定 scale 量化。
    mse01 = float(np.mean((restored01 - values01) ** 2))  # 计算反量化均方误差。
    baseline_rows01.append((batch01["step"], round(static_scale01, 6), round(saturation01, 3), round(mse01, 4)))  # 保存当前步结果。
print("固定Scale基线：step | scale | saturation | mse")  # 输出基线结果表头。
for row01 in baseline_rows01:  # 逐步展示饱和与误差。
    print(row01)  # 输出一行固定 scale 指标。


固定Scale基线：step | scale | saturation | mse
(1, 0.005357, 0.0, 0.0007)
(2, 0.005357, 0.333, 0.042)
(3, 0.005357, 0.333, 0.2179)
(4, 0.005357, 0.333, 202.0542)
(5, 0.005357, 0.333, 0.102)
(6, 0.005357, 0.333, 0.3686)


## 3. 核心实现：Amax History、Margin 与峰值 Fallback

每步先用最近三步历史 amax 计算 delayed scale，并保留 1 bit margin。若当前批次出现任何饱和，则立即用当前 amax 重算一次；这展示 delayed recipe 的状态与峰值修正。


In [3]:
history01 = [first_amax01]  # 初始化 delayed scaling 的 amax 历史。
delayed_rows01 = []  # 收集每步 scale、fallback 和误差。
for batch01 in batches01:  # 按训练顺序处理激活批次。
    values01 = np.array(batch01["values"], dtype=np.float32)  # 读取当前激活张量。
    historical_amax01 = max(history01[-3:])  # 取最近三步历史最大值。
    delayed_scale01 = historical_amax01 / (448.0 / 2.0)  # 预留一位 margin 计算 delayed scale。
    restored01, saturation01 = fake_fp8_01(values01, delayed_scale01)  # 使用历史 scale 尝试量化。
    fallback01 = saturation01 > 0.0  # 检查当前峰值是否超出历史范围。
    if fallback01:  # 对突发峰值执行当前批次重标定。
        current_amax01 = float(np.abs(values01).max())  # 读取当前批次 amax。
        delayed_scale01 = current_amax01 / (448.0 / 2.0)  # 使用当前 amax 和 margin 重算 scale。
        restored01, saturation01 = fake_fp8_01(values01, delayed_scale01)  # 用 fallback scale 再量化一次。
    mse01 = float(np.mean((restored01 - values01) ** 2))  # 计算最终反量化误差。
    delayed_rows01.append((batch01["step"], round(historical_amax01, 2), round(delayed_scale01, 6), fallback01, round(saturation01, 3), round(mse01, 4)))  # 保存 scale 状态轨迹。
    history01.append(float(np.abs(values01).max()))  # 在本步结束后更新 amax history。
print("Delayed Scaling：step | history_amax | scale | fallback | saturation | mse")  # 输出核心过程表头。
for row01 in delayed_rows01:  # 逐步展示 scale 状态变化。
    print(row01)  # 输出当前批次的历史与修正结果。


Delayed Scaling：step | history_amax | scale | fallback | saturation | mse
(1, 2.4, 0.010714, False, 0.0, 0.0007)
(2, 2.4, 0.010714, False, 0.0, 0.0012)
(3, 2.8, 0.0125, False, 0.0, 0.0033)
(4, 3.3, 0.125, True, 0.0, 0.0004)
(5, 28.0, 0.125, False, 0.0, 0.0018)
(6, 28.0, 0.125, False, 0.0, 0.0019)


## 4. 结果表与结果解读

固定 scale 在第四步明显饱和；delayed recipe 先检测到历史滞后，再用当前 amax 修正，因此最终饱和率为零。Margin 扩大了范围但也会牺牲小值精度，真实 recipe 必须按层校准。


In [4]:
baseline_saturation01 = sum(row01[2] for row01 in baseline_rows01)  # 汇总固定 scale 饱和率。
delayed_saturation01 = sum(row01[4] for row01 in delayed_rows01)  # 汇总 fallback 后饱和率。
baseline_mse01 = sum(row01[3] for row01 in baseline_rows01) / len(baseline_rows01)  # 计算固定 scale 平均 MSE。
delayed_mse01 = sum(row01[5] for row01 in delayed_rows01) / len(delayed_rows01)  # 计算 delayed recipe 平均 MSE。
print("方法 | 累计饱和率 | 平均MSE | 峰值处理")  # 输出两种方法对照表头。
print("static_first_batch", round(baseline_saturation01, 3), round(baseline_mse01, 4), "clip")  # 展示固定 scale 结果。
print("delayed_plus_fallback", round(delayed_saturation01, 3), round(delayed_mse01, 4), "rescale")  # 展示历史加峰值修正结果。
print("结果解读：Delayed Scaling减少scale抖动，但峰值检测和饱和遥测不可省略")  # 解释历史配方与异常保护的关系。


方法 | 累计饱和率 | 平均MSE | 峰值处理
static_first_batch 1.665 33.7976 clip
delayed_plus_fallback 0.0 0.0016 rescale
结果解读：Delayed Scaling减少scale抖动，但峰值检测和饱和遥测不可省略


## 5. 失败案例与修正：只依赖历史导致突发峰值裁剪

第四步的 28 超出最近历史，纯 delayed scale 会裁剪。修正不是永久改用大范围 E5M2，而是先依据张量角色选择格式，再对偶发峰值做受控 fallback。


In [5]:
spike01 = np.array(batches01[3]["values"], dtype=np.float32)  # 读取异常峰值批次。
pure_delayed_scale01 = max(history01[:3]) / (448.0 / 2.0)  # 模拟只看峰值前历史的 scale。
pure_restored01, pure_saturation01 = fake_fp8_01(spike01, pure_delayed_scale01)  # 执行没有 fallback 的历史量化。
fixed_scale01 = float(np.abs(spike01).max()) / (448.0 / 2.0)  # 使用当前批次计算修正 scale。
fixed_restored01, fixed_saturation01 = fake_fp8_01(spike01, fixed_scale01)  # 执行峰值 fallback 量化。
print("失败行为：纯历史scale", round(pure_delayed_scale01, 6), "饱和率", round(pure_saturation01, 3), "最大恢复值", round(float(np.abs(pure_restored01).max()), 2))  # 展示历史滞后裁剪。
print("修正行为：当前amax fallback", round(fixed_scale01, 6), "饱和率", round(fixed_saturation01, 3), "最大恢复值", round(float(np.abs(fixed_restored01).max()), 2))  # 展示峰值修正。


失败行为：纯历史scale 0.0125 饱和率 0.333 最大恢复值 5.6
修正行为：当前amax fallback 0.125 饱和率 0.0 最大恢复值 28.0


## 6. 生产边界与数值制品

真实 FP8 还涉及 E4M3/E5M2 编码细节、随机舍入、每 tensor/channel scale、反向梯度和硬件 kernel。训练必须保留高精度主权重，并把格式、历史长度、margin 和饱和阈值随 checkpoint 发布。


In [6]:
fp8_contract01 = {"forward": "E4M3", "gradient": "E5M2", "history": 3, "margin_bits": 1, "fallback": "current_amax_on_saturation", "master_weight": "FP32"}  # 定义训练数值发布合同。
print("FP8 配方制品", fp8_contract01)  # 展示格式与 scale recipe 的版本字段。
print("生产替换点：真实FP8编码、Tensor Core kernel、分布式amax同步、逐层校准和溢出告警")  # 说明教学量化器的边界。


FP8 配方制品 {'forward': 'E4M3', 'gradient': 'E5M2', 'history': 3, 'margin_bits': 1, 'fallback': 'current_amax_on_saturation', 'master_weight': 'FP32'}
生产替换点：真实FP8编码、Tensor Core kernel、分布式amax同步、逐层校准和溢出告警


## 7. 最小回归测试

断言保护案例规模、峰值反例和 fallback 结果。


In [7]:
assert len(batches01) >= 5  # 保证案例仍包含连续训练批次。
assert pure_saturation01 > 0.0  # 保证历史滞后的峰值失败可以复现。
assert fixed_saturation01 == 0.0  # 保证当前 amax fallback 消除饱和。
assert delayed_saturation01 < baseline_saturation01  # 保证核心 recipe 比固定 scale 更少饱和。
assert fp8_contract01["master_weight"] == "FP32"  # 保证高精度主权重边界不会丢失。
print("最小回归测试通过：历史scale、峰值fallback和主权重合同稳定")  # 显示 FP8 关键性质已验证。


最小回归测试通过：历史scale、峰值fallback和主权重合同稳定
